# Chronic Kidney Disease (CKD) Feature Selection and Evaluation

### Repository setup
This repository version uses the files under `data/` directly. The original Colab Drive mount has been removed for portability.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import (
    chi2, SelectKBest, f_classif, mutual_info_classif, RFE
)
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
df = pd.read_csv("/content/drive/MyDrive/CKD/kidney_disease.csv")

In [4]:
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

In [5]:
target_col = 'classification'

In [6]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [7]:
X = df.drop(columns=[target_col])
y = df[target_col]

In [8]:
# Scale features
X = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
display(X.head())

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane
0,-1.727726,-0.203139,0.258373,0.454071,-0.012548,-0.410106,-0.560523,0.684468,-0.291825,-0.17932,...,0.988022,0.204213,0.269257,-0.115612,1.298336,1.089659,-0.215922,-0.510723,-0.472690,-0.407926
1,-1.719066,-2.594124,-1.936857,0.454071,2.208413,-0.410106,-0.560523,0.684468,-0.291825,-0.17932,...,-0.421688,-0.366347,-0.299846,1.076262,-0.738270,-0.423756,-0.215922,-0.510723,-0.472690,-0.407926
2,-1.710406,0.613295,0.258373,-1.297699,0.727772,2.323069,0.895381,0.684468,-0.291825,-0.17932,...,-1.006202,-1.032001,0.198119,1.076262,-0.738270,1.089659,-0.215922,1.965511,-0.472690,2.357675
3,-1.701745,-0.203139,-0.473370,-2.173584,2.208413,-0.410106,0.895381,-1.838915,2.780013,-0.17932,...,-0.456071,-0.936908,-0.086433,-1.148569,1.298336,-0.423756,-0.215922,1.965511,2.048324,2.357675
4,-1.693085,-0.028189,0.258373,-1.297699,0.727772,-0.410106,0.895381,0.684468,-0.291825,-0.17932,...,-0.318538,-0.651628,0.126981,-0.592361,-0.738270,-0.423756,-0.215922,-0.510723,-0.472690,-0.407926


In [9]:
from sklearn.impute import SimpleImputer

# Impute missing values with the mean
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X_imputed = imputer.fit_transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=X.columns)



# 2. Filter Method
# Use ANOVA F-test for feature importance
filter_selector = SelectKBest(score_func=f_classif, k=12)
X_filter = filter_selector.fit_transform(X_imputed, y)
filter_features = X.columns[filter_selector.get_support()].tolist()
print("Filter Selected Features:", filter_features)


# 3. Wrapper Method (using RFE + Logistic Regression)

model = LogisticRegression(max_iter=1000)
rfe = RFE(model, n_features_to_select=10)
X_wrapper = rfe.fit_transform(X_imputed, y)
wrapper_features = X.columns[rfe.support_].tolist()
print("Wrapper Selected Features:", wrapper_features)


# 4. Embedded Method (using Random Forest)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_imputed, y)
importances = pd.Series(rf.feature_importances_, index=X.columns)
embedded_features = importances.sort_values(ascending=False).head(9).index.tolist()
print("Embedded Selected Features:", embedded_features)


# 5. Intersection of all subsets

final_features = list(set(filter_features) & set(wrapper_features) & set(embedded_features))
print("Final Intersection Features:", final_features)

Filter Selected Features: ['id', 'sg', 'al', 'rbc', 'pc', 'bgr', 'bu', 'hemo', 'pcv', 'htn', 'appet', 'pe']
Wrapper Selected Features: ['id', 'sg', 'al', 'rbc', 'bgr', 'sod', 'hemo', 'htn', 'appet', 'pe']
Embedded Selected Features: ['id', 'hemo', 'sg', 'sc', 'al', 'rbc', 'htn', 'pcv', 'bgr']
Final Intersection Features: ['hemo', 'sg', 'htn', 'rbc', 'id', 'bgr', 'al']


In [10]:

# 6. Model Evaluation Function

def evaluate_subset(X_subset, y):
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.3, random_state=42)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=0),
        "Recall": recall_score(y_test, y_pred, pos_label=0),
        "F1-Score": f1_score(y_test, y_pred, pos_label=0)
    }


# 7. Evaluate all subsets

results = {}

datasets = {
    "Filter": X_imputed[filter_features],
    "Wrapper": X_imputed[wrapper_features],
    "Embedded": X_imputed[embedded_features],
    "Final (Intersection)": X_imputed[final_features] if final_features else None
}

for name, X_sub in datasets.items():
    if X_sub is not None and X_sub.shape[1] > 0:
        results[name] = evaluate_subset(X_sub, y)

# Display results
print("\nModel Evaluation Results:")
for method, scores in results.items():
    print(f"\n{method} Method:")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")


Model Evaluation Results:

Filter Method:
Accuracy: 0.9917
Precision: 1.0000
Recall: 0.9773
F1-Score: 0.9885

Wrapper Method:
Accuracy: 0.9917
Precision: 1.0000
Recall: 0.9773
F1-Score: 0.9885

Embedded Method:
Accuracy: 0.9833
Precision: 0.9773
Recall: 0.9773
F1-Score: 0.9773

Final (Intersection) Method:
Accuracy: 0.9833
Precision: 0.9773
Recall: 0.9773
F1-Score: 0.9773


In [11]:
# Intersection of all subsets
final_features = list(set(filter_features) & set(wrapper_features) & set(embedded_features))
print("Final Intersection Features:", final_features)


Final Intersection Features: ['hemo', 'sg', 'htn', 'rbc', 'id', 'bgr', 'al']


In [12]:
features_df = pd.DataFrame({
    "Method": ["Filter", "Wrapper", "Embedded", "Final Intersection"],
    "Selected Features": [
        ", ".join(filter_features),
        ", ".join(wrapper_features),
        ", ".join(embedded_features),
        ", ".join(final_features) if final_features else "None"
    ]
})
features_df.to_csv("selected_features.csv", index=False)
print("✅ Saved: selected_features.csv")

✅ Saved: selected_features.csv


In [13]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Method"})
results_df.to_csv("model_results.csv", index=False)
print("✅ Saved: model_results.csv")

✅ Saved: model_results.csv


In [14]:
if final_features:
    intersection_df = df[final_features + [target_col]]
    intersection_df.to_csv("final_intersection_dataset.csv", index=False)
    print("✅ Saved: final_intersection_dataset.csv")

✅ Saved: final_intersection_dataset.csv
